# Parte 2: Modelo Predictivo de Churn
## Data Scientist Challenge – Entel

**Restricciones de negocio:**
- Solo se pueden contactar **2.000 clientes/mes**
- Costo por contacto: **$1.000 CLP**
- Valor recuperado por cliente retenido: **$30.000 CLP**
- Tasa de éxito de retención: **10%** sobre los churners contactados

**Lógica de ganancia:**
`Ganancia = (True Positives × 0.10 × $30.000) – (2.000 × $1.000)`

---

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

SEED = 42
np.random.seed(SEED)

# Constantes de negocio
BUDGET_CONTACTS   = 2_000       # clientes contactables por mes
COST_PER_CONTACT  = 1_000       # CLP por contacto
VALUE_RETAINED    = 30_000      # CLP por cliente retenido
RETENTION_RATE    = 0.10        # 10% de éxito solo sobre churners reales

DATA_PATH = '../documentos/dataset_prueba.csv'

In [4]:
df = pd.read_csv(DATA_PATH)
df['last_date_of_month'] = pd.to_datetime(df['last_date_of_month'])

def crear_features(data):
    d = data.copy()
    total_vol = d['vol_2g_mb'].fillna(0) + d['vol_3g_mb'].fillna(0)
    d['ratio_3g_vs_total']   = np.where(total_vol > 0, d['vol_3g_mb'].fillna(0) / total_vol, 0)
    d['total_voz_mou']       = d['total_og_mou'].fillna(0) + d['total_ic_mou'].fillna(0)
    roam_total = d['roam_ic_mou'].fillna(0) + d['roam_og_mou'].fillna(0)
    d['ratio_roam']          = np.where(d['total_voz_mou'] > 0, roam_total / d['total_voz_mou'], 0)
    d['hizo_recarga_datos']  = (d['total_rech_data'].fillna(0) > 0).astype(int)
    arpu_mean                = d['arpu'].mean()
    d['arpu_vs_media']       = d['arpu'] - arpu_mean
    d['avg_rech_amt']        = np.where(d['total_rech_num'] > 0, d['total_rech_amt'] / d['total_rech_num'], 0)
    d['es_usuario_datos']    = (total_vol > 0).astype(int)
    d['balance_og_ic']       = d['total_og_mou'].fillna(0) - d['total_ic_mou'].fillna(0)
    d['inactivo_total']      = (
        (d['total_og_mou'].fillna(0) == 0) &
        (d['total_ic_mou'].fillna(0) == 0) &
        (d['total_rech_amt'].fillna(0) == 0)
    ).astype(int)
    return d

df = crear_features(df)

train_dates = pd.to_datetime(['2014-06-30', '2014-07-31'])
test_date   = pd.to_datetime('2014-08-31')

df_train = df[df['last_date_of_month'].isin(train_dates)].copy()
df_test  = df[df['last_date_of_month'] == test_date].copy()

print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')
print(f'Churn en train: {df_train["churn"].mean()*100:.2f}%  |  Churn en test: {df_test["churn"].mean()*100:.2f}%')

Train: 199,397 | Test: 98,899
Churn en train: 1.28%  |  Churn en test: 1.90%


## 1. Preparación de features para modelado

Se excluyen columnas no predictivas (IDs, fechas, target) y se realiza imputación con 0 para variables de uso.

In [5]:
# Columnas a excluir del modelo
EXCLUDE_COLS = [
    'mobile_number', 'last_date_of_month', 'churn',
    'date_of_last_rech', 'date_of_last_rech_data',
    'arpu_segment'  # Variable categórica derivada — se podría one-hot, pero se omite para simplicidad
]

# Seleccionar solo columnas numéricas
feature_cols = [
    c for c in df_train.select_dtypes(include=np.number).columns
    if c not in EXCLUDE_COLS
]

print(f'Total de features: {len(feature_cols)}')
print(feature_cols)

X_train = df_train[feature_cols].copy()
y_train = df_train['churn'].values

X_test  = df_test[feature_cols].copy()
y_test  = df_test['churn'].values

# Imputar nulos con 0 (uso ausente = no uso ese mes)
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

print(f'\nNulos restantes en X_train: {X_train.isnull().sum().sum()}')
print(f'Nulos restantes en X_test:  {X_test.isnull().sum().sum()}')

Total de features: 60
['arpu', 'onnet_mou', 'offnet_mou', 'roam_ic_mou', 'roam_og_mou', 'loc_og_t2t_mou', 'loc_og_t2m_mou', 'loc_og_t2f_mou', 'loc_og_t2c_mou', 'loc_og_mou', 'std_og_t2t_mou', 'std_og_t2m_mou', 'std_og_t2f_mou', 'std_og_t2c_mou', 'std_og_mou', 'isd_og_mou', 'spl_og_mou', 'og_others', 'total_og_mou', 'loc_ic_t2t_mou', 'loc_ic_t2m_mou', 'loc_ic_t2f_mou', 'loc_ic_mou', 'std_ic_t2t_mou', 'std_ic_t2m_mou', 'std_ic_t2f_mou', 'std_ic_t2o_mou', 'std_ic_mou', 'total_ic_mou', 'spl_ic_mou', 'isd_ic_mou', 'ic_others', 'total_rech_num', 'total_rech_amt', 'max_rech_amt', 'last_day_rch_amt', 'total_rech_data', 'max_rech_data', 'count_rech_2g', 'count_rech_3g', 'av_rech_amt_data', 'vol_2g_mb', 'vol_3g_mb', 'arpu_3g', 'arpu_2g', 'night_pck_user', 'monthly_2g', 'sachet_2g', 'monthly_3g', 'sachet_3g', 'fb_user', 'ratio_3g_vs_total', 'total_voz_mou', 'ratio_roam', 'hizo_recarga_datos', 'arpu_vs_media', 'avg_rech_amt', 'es_usuario_datos', 'balance_og_ic', 'inactivo_total']

Nulos restantes 

## 2. Función de métricas de negocio

Dado que el objetivo final es **maximizar la ganancia** dentro del presupuesto de 2.000 contactos, definimos una función que calcula el resultado económico dado un set de predicciones.

In [7]:
def ganancia_negocio(y_true, y_proba, top_n=BUDGET_CONTACTS,
                     cost=COST_PER_CONTACT, value=VALUE_RETAINED,
                     retention_rate=RETENTION_RATE):
    """
    Calcula la ganancia neta seleccionando los top_n clientes
    con mayor probabilidad de churn.

    Ganancia = TP_top_n * retention_rate * value - top_n * cost
    """
    df_eval = pd.DataFrame({'y_true': y_true, 'y_proba': y_proba})
    df_eval = df_eval.sort_values('y_proba', ascending=False).head(top_n)

    tp = df_eval['y_true'].sum()   # Verdaderos churners en el top N
    fp = top_n - tp                # No churners contactados (costo sin retorno)

    ingresos = tp * retention_rate * value
    costos   = top_n * cost
    ganancia = ingresos - costos

    return {
        'top_n': top_n,
        'true_positives': tp,
        'false_positives': fp,
        'precision_top_n': tp / top_n,
        'ingresos_CLP': ingresos,
        'costos_CLP': costos,
        'ganancia_neta_CLP': ganancia,
        'ROI': (ganancia / costos) * 100 if costos > 0 else 0
    }

def ganancia_random_baseline(y_true, top_n=BUDGET_CONTACTS,
                              cost=COST_PER_CONTACT, value=VALUE_RETAINED,
                              retention_rate=RETENTION_RATE):
    """Ganancia esperada si se seleccionan clientes aleatoriamente."""
    churn_rate = y_true.mean()
    tp_esperados = top_n * churn_rate
    ingresos = tp_esperados * retention_rate * value
    costos   = top_n * cost
    return {
        'tp_esperados': tp_esperados,
        'ingresos_CLP': ingresos,
        'costos_CLP': costos,
        'ganancia_neta_CLP': ingresos - costos
    }

baseline = ganancia_random_baseline(y_test)
print('=== Baseline aleatorio (test) ===')
for k, v in baseline.items():
    print(f'  {k}: {v:,.0f}')

=== Baseline aleatorio (test) ===
  tp_esperados: 38
  ingresos_CLP: 114,177
  costos_CLP: 2,000,000
  ganancia_neta_CLP: -1,885,823


## 3. Modelamiento

### Elección de modelos

Se entrenan 3 modelos con distintos trade-offs:

| Modelo | Ventajas | Desventajas |
|--------|----------|-------------|
| **Regresión Logística** | Interpretable, rápido, buen baseline | Puede no capturar no-linealidades |
| **Random Forest** | Robusto, maneja no-linealidades, importancia de features | Lento en datasets grandes |
| **LightGBM** | Muy eficiente, alta performance en datos tabulares, maneja desbalance | Menos interpretable |

**Decisión principal:** Dado el desbalance de clases (churn es minoría), se optimizan los modelos por **AUC-ROC** y se evalúan por **ganancia de negocio** en el top 2.000.

In [9]:
results = {}

# ============================================================
# MODELO 1: Regresión Logística (baseline interpretable)
# ============================================================
print('Entrenando Regresión Logística...')

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',   # Maneja desbalance de clases
        max_iter=500,
        random_state=SEED,
        C=0.1                      # Regularización para evitar overfitting
    ))
])

lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]

results['Logistic Regression'] = {
    'proba': lr_proba,
    'auc': roc_auc_score(y_test, lr_proba),
    'ap': average_precision_score(y_test, lr_proba),
    'negocio': ganancia_negocio(y_test, lr_proba)
}
print(f'  AUC-ROC: {results["Logistic Regression"]["auc"]:.4f}')
print(f'  Ganancia neta: ${results["Logistic Regression"]["negocio"]["ganancia_neta_CLP"]:,.0f} CLP')

Entrenando Regresión Logística...
  AUC-ROC: 0.8661
  Ganancia neta: $-821,000 CLP


In [10]:
# ============================================================
# MODELO 2: LightGBM (modelo principal — mejor balance performance/velocidad)
# ============================================================

# Calcular peso de clases para manejar desbalance
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f'  scale_pos_weight: {scale_pos_weight:.2f} (ratio no-churn / churn)')

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    scale_pos_weight=scale_pos_weight,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

lgb_proba = lgb_model.predict_proba(X_test)[:, 1]

results['LightGBM'] = {
    'proba': lgb_proba,
    'auc': roc_auc_score(y_test, lgb_proba),
    'ap': average_precision_score(y_test, lgb_proba),
    'negocio': ganancia_negocio(y_test, lgb_proba)
}
print(f'  AUC-ROC: {results["LightGBM"]["auc"]:.4f}')
print(f'  Ganancia neta: ${results["LightGBM"]["negocio"]["ganancia_neta_CLP"]:,.0f} CLP')

Entrenando LightGBM...
  scale_pos_weight: 77.16 (ratio no-churn / churn)
  AUC-ROC: 0.8526
  Ganancia neta: $-380,000 CLP


In [12]:
# ============================================================
# MODELO 3: Random Forest
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

results['Random Forest'] = {
    'proba': rf_proba,
    'auc': roc_auc_score(y_test, rf_proba),
    'ap': average_precision_score(y_test, rf_proba),
    'negocio': ganancia_negocio(y_test, rf_proba)
}
print(f'  AUC-ROC: {results["Random Forest"]["auc"]:.4f}')
print(f'  Ganancia neta: ${results["Random Forest"]["negocio"]["ganancia_neta_CLP"]:,.0f} CLP')

  AUC-ROC: 0.8832
  Ganancia neta: $157,000 CLP
